In [7]:
# Inline backend: figures show in the notebook (Agg hides them with only a warning on plt.show()).
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [4]:
SESSION = "20260422_201653"
STREAM = "head"  # "head" | "left" | "right"

# Optional extra crop on monotonic_ns after overlap trim (set both to None to skip)
ROI_MONO_MIN = None
ROI_MONO_MAX = None

_cam_joint_paths = {
    "head": (
        f"{SESSION}/head_apriltag_processed.csv",
        f"{SESSION}/joint_log_left.csv",
    ),
    "left": (
        f"{SESSION}/left_apriltag_processed.csv",
        f"{SESSION}/joint_log_left.csv",
    ),
    "right": (
        f"{SESSION}/right_apriltag_processed.csv",
        f"{SESSION}/joint_log_right.csv",
    ),
}
if STREAM not in _cam_joint_paths:
    raise ValueError(f"STREAM must be one of {list(_cam_joint_paths)}, got {STREAM!r}")

cam_path, joint_path = _cam_joint_paths[STREAM]
cam_data = pd.read_csv(cam_path)
joint_data = pd.read_csv(joint_path)

joint_data = joint_data[["monotonic_ns", "q5"]].copy()
joint_data["q5"] = pd.to_numeric(joint_data["q5"], errors="coerce")
joint_zero = joint_data["q5"].iloc[0]
joint_data["joint_ang"] = -(joint_data["q5"] - joint_zero) * 180.0 / np.pi

cam_data = cam_data.copy()
cam_data["vector_x"] = pd.to_numeric(cam_data["vector_x"], errors="coerce")
cam_data["vector_y"] = pd.to_numeric(cam_data["vector_y"], errors="coerce")
first_cam_angle = np.arctan2(cam_data.loc[0, "vector_y"], cam_data.loc[0, "vector_x"])
cam_data["cam_ang"] = np.degrees(
    np.arctan2(cam_data["vector_y"], cam_data["vector_x"]) - first_cam_angle
)
cam_data["cam_ang"] = ((cam_data["cam_ang"] + 180) % 360) - 180

if "frame_time_ns" not in cam_data.columns:
    if "presentation_time_ns" in cam_data.columns:
        cam_data["frame_time_ns"] = pd.to_numeric(
            cam_data["presentation_time_ns"], errors="coerce"
        )
    else:
        cam_data["frame_time_ns"] = pd.to_numeric(cam_data["monotonic_ns"], errors="coerce")

merged = pd.merge(cam_data, joint_data, on="monotonic_ns", how="outer", suffixes=("_cam", "_joint"))
merged = merged.sort_values("monotonic_ns").reset_index(drop=True)
merged = merged.bfill()
merged = merged[["monotonic_ns", "frame_time_ns", "cam_ang", "joint_ang"]]

overlap_start = max(
    int(cam_data["monotonic_ns"].min()), int(joint_data["monotonic_ns"].min())
)
overlap_end = min(
    int(cam_data["monotonic_ns"].max()), int(joint_data["monotonic_ns"].max())
)
merged = merged[
    (merged["monotonic_ns"] >= overlap_start) & (merged["monotonic_ns"] <= overlap_end)
].reset_index(drop=True)

if ROI_MONO_MIN is not None:
    merged = merged[merged["monotonic_ns"] >= ROI_MONO_MIN]
if ROI_MONO_MAX is not None:
    merged = merged[merged["monotonic_ns"] <= ROI_MONO_MAX]

merged = merged.reset_index(drop=True)
merged.head()


,monotonic_ns,frame_time_ns,cam_ang,joint_ang
0,19297167044181,1.929717e+13,0.000000,-15.059451
1,19297171515064,1.929720e+13,0.682326,-15.059451
2,19297179525910,1.929720e+13,0.682326,-14.884183
3,19297187518467,1.929720e+13,0.682326,-14.707082
4,19297195519730,1.929720e+13,0.682326,-14.525053


In [8]:
out_csv = f"{SESSION}/merged_output_{STREAM}.csv"
merged.to_csv(out_csv, index=False)
print(f"Exported {out_csv}")


Exported 20260422_201653/merged_output_head.csv


In [9]:
plt.figure(figsize=(10, 5))
t = merged["monotonic_ns"].to_numpy()
plt.plot(t, merged["cam_ang"].to_numpy(), label="cam_ang", color="blue")
plt.plot(t, merged["joint_ang"].to_numpy(), label="joint_ang", color="orange")
plt.xlabel("monotonic_ns")
plt.ylabel("angle (deg)")
plt.legend()
plt.title(f"Camera vs joint ({STREAM})")
plt.tight_layout()
plt.show()


/tmp/ipykernel_397891/2906890791.py:10: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [ ]:
from scipy.signal import savgol_filter

# --- Resample to uniform time grid ---
clean = merged[["monotonic_ns", "cam_ang", "joint_ang"]].dropna()
times = clean["monotonic_ns"].to_numpy(dtype=np.float64)
cam = clean["cam_ang"].to_numpy(dtype=np.float64)
joint = clean["joint_ang"].to_numpy(dtype=np.float64)

dt = np.median(np.diff(times))
uniform_times = np.arange(times[0], times[-1] + dt, dt)
cam_r = np.interp(uniform_times, times, cam)
joint_r = np.interp(uniform_times, times, joint)

# --- Smoothing (Savitzky-Golay) ---
win = max(11, int(len(uniform_times) * 0.01) | 1)
poly = 3
cam_s = savgol_filter(cam_r, window_length=win, polyorder=poly)
joint_s = savgol_filter(joint_r, window_length=win, polyorder=poly)

cam_d1 = np.gradient(cam_s, dt)
joint_d1 = np.gradient(joint_s, dt)


def zero_crossing_times(signal, t):
    """Return times of sign changes (zero crossings) in signal."""
    signs = np.sign(signal)
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    cross_idx = np.where(np.diff(signs))[0]
    zc_times = []
    for idx in cross_idx:
        t0, t1 = t[idx], t[idx + 1]
        s0, s1 = signal[idx], signal[idx + 1]
        if s1 != s0:
            zc_times.append(t0 - s0 * (t1 - t0) / (s1 - s0))
    return np.array(zc_times)


cam_zc = zero_crossing_times(cam_d1, uniform_times)
joint_zc = zero_crossing_times(joint_d1, uniform_times)

print(f"cam  derivative zero crossings: {len(cam_zc)}")
print(f"joint derivative zero crossings: {len(joint_zc)}")

lags = []
for t_cam in cam_zc:
    diffs = joint_zc - t_cam
    nearest = diffs[np.argmin(np.abs(diffs))]
    lags.append(nearest)

lags = np.array(lags)
median_lag_ns = np.median(lags)
median_lag_s = median_lag_ns * 1e-9

print(f"\nMedian lag per zero-crossing pair: {median_lag_ns:.0f} ns  ({median_lag_s*1000:.3f} ms)")
print("Positive  → joint lags cam  (cam leads)")
print("Negative  → cam  lags joint (joint leads)")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(uniform_times, cam_s, label="cam_ang (smoothed)", color="blue")
axes[0].plot(uniform_times, joint_s, label="joint_ang (smoothed)", color="orange")
axes[0].vlines(cam_zc, *axes[0].get_ylim(), color="blue", alpha=0.3, linewidth=0.8, label="cam deriv ZC")
axes[0].vlines(
    joint_zc, *axes[0].get_ylim(), color="orange", alpha=0.3, linewidth=0.8, label="joint deriv ZC"
)
axes[0].set_ylabel("Angle (deg)")
axes[0].set_title(f"Smoothed signals + derivative zero crossings ({STREAM})")
axes[0].legend()

axes[1].plot(uniform_times, cam_d1, label="cam_ang'", color="blue")
axes[1].plot(uniform_times, joint_d1, label="joint_ang'", color="orange")
axes[1].axhline(0, color="k", linewidth=0.5)
axes[1].vlines(cam_zc, axes[1].get_ylim()[0], axes[1].get_ylim()[1], color="blue", alpha=0.3, linewidth=0.8)
axes[1].vlines(
    joint_zc, axes[1].get_ylim()[0], axes[1].get_ylim()[1], color="orange", alpha=0.3, linewidth=0.8
)
axes[1].set_ylabel("dAngle/dt  (deg/ns)")
axes[1].set_xlabel("monotonic_ns")
axes[1].set_title("1st derivatives")
axes[1].legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(lags * 1e-6, bins=30, color="steelblue", edgecolor="white")
plt.axvline(
    median_lag_ns * 1e-6,
    color="red",
    linestyle="--",
    label=f"Median = {median_lag_ns*1e-6:.2f} ms",
)
plt.xlabel("Lag (ms)")
plt.ylabel("Count")
plt.title(f"Per-peak phase lags ({STREAM})")
plt.legend()
plt.tight_layout()
plt.show()
